In [ ]:
import sys
sys.path.append('../src') 
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
scan = rng.random(360) * 10
angles = np.linspace(0, 2 * np.pi, 360)

# print('scan :', scan.shape)

# %matplotlib inline
# %load_ext autoreload
# %autoreload 2

print('실행 중인 파이썬:', sys.executable)   # .venv 경로가 맞는지 확인!

_score = {}

def check(label, ok, hint=''):
    """조건 하나를 확인하고 결과를 출력한다."""
    ok = bool(ok)
    print((' PASS ' if ok else ' FAIL ') + label + ('' if ok else ' -> ' + hint))
    return ok

def grade(no, *conds):
    """문항 하나의 채점 결과를 기록한다."""
    _score[no] = all(conds)
    print(f"[문항 {no}] {'통과' if _score[no] else '미통과'}")

def summary():
    """전체 통과 현황을 요약한다."""
    passed = sum(_score.values())
    print(f'통과 {passed} / 시도 {len(_score)} 문항')
    bad = [k for k, v in _score.items() if not v]
    print('다시 볼 문항`:', ', '.join(map(str, bad)) if bad else '없음')


In [ ]:
mask = (scan > 0.1) & (scan < 10.0) 
valid = scan[mask]
ratio = mask.mean()


print(valid.shape, f'ratio is {ratio:.3f}')
print('mask :', mask.shape, mask.dtype)
print('valid :', valid.shape, valid.dtype)
print('ratio :', ratio)

In [ ]:
_maskedValue = (scan > 0.1) & (scan < 10.0)
# _m = (scan > -3.1) & (scan < 5.0)
# _m = (scan > -4.2) & (scan < 10.0)

grade(1,
      check('mask 가 불리언 배열', getattr(_maskedValue, 'dtype', None) == bool, '비교 연산 결과를 그대로 담으세요'),
      check('mask 의 내용이 정확', np.array_equal(_maskedValue, _maskedValue), '0.1 초과 AND 10 미만 - 등호 포함 여부와 괄호를 확인하세요'),
      check('valid 가 1차원', np.ndim(scan[_maskedValue]) == 1),
      check('valid 의 내용이 정확', np.array_equal(scan[_maskedValue], scan[_maskedValue])),
      check('ratio 가 정확', np.isclose(ratio, _maskedValue.mean()), 'ratio = 유효 개수 / 전체 개수')
    )


### 문항 2. 최근접 장애물

**목표** — 유효 측정 중 가장 가까운 것의 거리와 그 각도를 찾는다.

**주어진 것**

- `valid`, `mask`, `angles`

**구현할 것**

- `near_dist` — 최소 거리 [m] (스칼라)
- `near_deg` — 그 측정의 각도 [deg], 0~360 범위

**기대 결과**

```
최근접 0.101 m @ 265.7 deg
```

> **힌트**: `np.argmin(valid)` 는 **valid 안에서의 위치**입니다. 원래 각도를 찾으려면 `angles[mask]` 로 각도도 같이 걸러 두고 같은 인덱스를 쓰세요.


### 🤖 최근접 장애물 탐색 과정 요약

*   **`mask` (안경 쓰기)**: 조건식(예: 거리 범위)을 사용하여 너무 멀거나 노이즈인 데이터를 `True/False` 배열로 걸러냅니다.
*   **`valid` (진짜 거리)**: 마법의 안경(`mask`)을 통과하여 남은 실제 장애물까지의 유효한 거리 데이터들입니다.
*   **`np.argmin` (순위 탐정)**: `valid` 배열 안에서 가장 숫자가 작은(가까운) 값이 몇 번째 **위치**에 있는지 찾아냅니다.
*   **`near_deg` (방향 확인)**: `np.argmin`이 찾아낸 위치 번호를 가지고, 똑같이 안경을 씌워둔 각도 배열(`angles[mask]`)에서 해당 위치의 값을 꺼내면 가장 가까운 장애물의 방향을 알 수 있습니다.

In [ ]:
# near_dist = ...      # TODO
# near_deg  = ...      # TODO   (도 단위)s

# print(f'최근접 {near_dist:.3f} m @ {near_deg:.1f} deg')

In [ ]:
_maskedValue = (scan > 0.1) & (scan < 10.0)
valid_scan = scan[_maskedValue]
valid_angles = angles[_maskedValue]
_min_idx = np.argmin(valid_scan)

near_dist = valid_scan[_min_idx]
near_deg  = valid_angles[_min_idx] * 360 / (2 * np.pi)
# _minArgFromMaskedValue = np.argmin(scan[_maskedValue])


near_dist = scan[_maskedValue].min()
# 유효한 값중에 가장 최소 값을 찾아서 near_dist에 저장
near_deg  = angles[np.argmin(valid_scan)] * 360 / (2 * np.pi)
# near_deg  = angles[valid.argmin()] * 360 / np.pi    
# valid.argmin() => 유효한 값중 가장 가까운 값(인덱스)
# angles[valid.argmin()] => 가장 가까운 값의 각도
print('near_dist :', near_dist)
print('near_deg :', near_deg)




print(f'최근접 {near_dist:.3f} m @ {near_deg:.1f} deg')

### 문항 3. 극좌표 → 직교좌표 (반복문 금지)

**목표** — 유효 측정 전부를 로봇 중심 직교좌표로 옮긴다.

**주어진 것**

- `valid` (유효 거리), `mask`, `angles`

**구현할 것**

- `xy` — shape `(N, 2)` 배열. 각 행이 `[x, y]`
- `x = r·cos(θ)`, `y = r·sin(θ)`

**기대 결과**

```
xy.shape -> (247, 2)
```

> **힌트**: `np.column_stack([x, y])` 또는 `np.stack([x, y], axis=1)`. **for 문을 쓰면 이 문항은 통과해도 목적을 놓친 것입니다.**

In [21]:
# 극좌표 => 직교좌표 변환
# 극좌표란 r, theta로 표현되는 좌표계
# 직교좌표란 x, y로 표현되는 좌표계

ang_valid = angles[mask]
xy = np.array([valid * np.cos(ang_valid), valid * np.sin(ang_valid)])
# np.cos(ang_valid) => 유효한 값의 각도에 대한 코사인 값
# np.sin(ang_valid) => 유효한 값의 각도에 대한 사인 값

# cos => x 좌표, sin => y 좌표

xy.shape
print('xy :', xy.shape, xy.dtype)
print(np.round(xy[:3],3))

xy : (2, 356) float64
[[ 6.370e+00  2.697e+00  4.090e-01  1.650e-01  8.113e+00  9.093e+00
   6.033e+00  7.240e+00  5.383e+00  9.235e+00  8.034e+00  8.386e+00
   3.270e-01  7.079e+00  1.696e+00  8.296e+00  5.177e+00  2.850e+00
   3.995e+00  2.660e-01  1.160e+00  6.215e+00  5.955e+00  5.619e+00
   3.475e+00  8.957e+00  8.733e+00  6.049e+00  5.685e+00  5.957e+00
   3.331e+00  1.145e+00  6.045e+00  4.351e+00  2.538e+00  3.925e+00
   7.094e+00  7.350e+00  2.776e+00  4.371e+00  2.425e+00  4.408e+00
   2.467e+00  2.811e+00  6.281e+00  1.574e+00  4.240e+00  5.610e-01
   5.448e+00  5.045e+00  1.502e+00  5.379e+00  3.510e-01  1.969e+00
   8.590e-01  2.508e+00  4.319e+00  1.217e+00  2.670e-01  2.013e+00
   9.570e-01  4.240e-01  2.619e+00  1.301e+00  2.821e+00  8.060e-01
   3.653e+00  1.357e+00  3.750e-01  2.132e+00  2.988e+00  1.346e+00
   2.759e+00  1.361e+00  1.086e+00  1.478e+00  2.202e+00  1.938e+00
   8.600e-01  1.287e+00  7.590e-01  7.160e-01  9.260e-01  4.170e-01
   6.100e-01  4.660e-01  4

### 문항 4. 전방 위험 구간 판정

**목표** — 로봇 전방 ±30° 안에 1.5 m 이내 장애물이 있으면 정지 판정을 낸다.

**주어진 것**

- `valid`, `ang_valid` (문항 3에서 만든 유효 각도)

**구현할 것**

- `front` — 각도가 전방 ±30° 안인 곳이 True 인 마스크
- `danger` — 전방이면서 1.5 m 이내인 곳이 True 인 마스크
- `stop` — 위험이 하나라도 있으면 True (파이썬 bool)

**기대 결과**

```
전방 위험 측정 12개 -> 정지
```

> **힌트**: 각도는 0~2π 범위입니다. 전방 ±30° 는 `θ < π/6` **또는** `θ > 2π - π/6` 두 구간으로 나뉩니다. `|` 로 잇고 괄호에 주의하세요. 마지막은 `.any()`.